# DE-04 — Python & SQL Engineering

**Dataset:** `data/loan_data_04.csv`

This notebook builds a reusable CLI-style ETL flow using modular Python, parameterized SQL, transactions, configuration separation, structured exceptions, staging, merge/upsert, metadata columns, and row-count audit data.

The demonstration uses an in-memory SQLite database so it runs anywhere. The same engineering principles apply to PostgreSQL; production bulk loading would normally use PostgreSQL `COPY` or an equivalent driver API.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

DATA_FILE = ROOT / "data" / "loan_data_04.csv"
assert DATA_FILE.exists(), f"Dataset not found: {DATA_FILE}"

raw = pd.read_csv(DATA_FILE)
print(f"Dataset: {DATA_FILE.name}")
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
print(raw.head(3).to_string(index=False))

Dataset: loan_data_04.csv
Rows: 48 | Columns: 13
 Loan_ID Gender Married Dependents Education Self_Employed  ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  Credit_History Property_Area Loan_Status
LP001778   Male     Yes          1  Graduate            No             3155             1779.0       140.0             360.0             1.0     Semiurban           Y
LP001788 Female      No          0  Graduate           Yes             3463                0.0       122.0             360.0             NaN         Urban           Y
LP001790 Female      No          1  Graduate            No             3812                0.0       112.0             360.0             1.0         Rural           Y


## Learning Content

- Use bind parameters; never construct SQL from untrusted values.
- Keep a related write set inside one transaction.
- Separate extract, transform, load, and validation functions.
- Load environment-specific configuration outside source code.
- Raise structured exceptions and return meaningful CLI exit codes.
- Prefer bulk loading into staging, then merge into the target.
- Attach `run_id`, source file, and timestamps for auditability.

In [2]:
import os
import sqlite3
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone

class PipelineError(RuntimeError):
    """Expected pipeline failure."""

class ContractError(PipelineError):
    """Source contract failure."""

@dataclass(frozen=True)
class AppConfig:
    environment: str
    batch_size: int

def load_config() -> AppConfig:
    # Non-secret defaults are acceptable. Secrets must come from a secret manager
    # or environment and are deliberately not printed.
    return AppConfig(
        environment=os.getenv("APP_ENV", "training"),
        batch_size=int(os.getenv("BATCH_SIZE", "1000")),
    )

REQUIRED_COLUMNS = {"Loan_ID", "ApplicantIncome", "CoapplicantIncome",
                    "LoanAmount", "Loan_Status"}

In [3]:
def extract(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    missing = REQUIRED_COLUMNS.difference(frame.columns)
    if missing:
        raise ContractError(f"Missing required columns: {sorted(missing)}")
    return frame

def transform(frame: pd.DataFrame, run_id: str) -> pd.DataFrame:
    result = frame.copy()
    result["Loan_ID"] = result["Loan_ID"].astype("string").str.strip()
    for column in ["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]:
        result[column] = pd.to_numeric(result[column], errors="coerce")
    result["TotalIncome"] = result["ApplicantIncome"] + result["CoapplicantIncome"]
    result["run_id"] = run_id
    result["source_file"] = DATA_FILE.name
    result["loaded_at"] = datetime.now(timezone.utc).isoformat()
    return result.drop_duplicates("Loan_ID", keep="last")

def selected_records(frame: pd.DataFrame):
    columns = ["Loan_ID", "ApplicantIncome", "CoapplicantIncome", "LoanAmount",
               "Loan_Status", "TotalIncome", "run_id", "source_file", "loaded_at"]
    return list(frame[columns].itertuples(index=False, name=None))

## Hands-on / Demonstration

### Build a reusable pipeline with staging and transactional merge

In [4]:
config = load_config()
run_id = str(uuid.uuid4())
transformed = transform(extract(DATA_FILE), run_id)

connection = sqlite3.connect(":memory:")
connection.execute("""
CREATE TABLE loan_target (
    Loan_ID TEXT PRIMARY KEY,
    ApplicantIncome REAL,
    CoapplicantIncome REAL,
    LoanAmount REAL,
    Loan_Status TEXT,
    TotalIncome REAL,
    run_id TEXT,
    source_file TEXT,
    loaded_at TEXT
)
""")
connection.execute("""
CREATE TABLE loan_stage (
    Loan_ID TEXT,
    ApplicantIncome REAL,
    CoapplicantIncome REAL,
    LoanAmount REAL,
    Loan_Status TEXT,
    TotalIncome REAL,
    run_id TEXT,
    source_file TEXT,
    loaded_at TEXT
)
""")

insert_stage_sql = """
INSERT INTO loan_stage VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

# executemany is a portable bulk-loading demonstration.
# PostgreSQL production code should prefer COPY for large batches.
try:
    with connection:
        connection.executemany(insert_stage_sql, selected_records(transformed))
        connection.execute("""
        INSERT INTO loan_target
        SELECT * FROM loan_stage
        WHERE 1
        ON CONFLICT(Loan_ID) DO UPDATE SET
            ApplicantIncome = excluded.ApplicantIncome,
            CoapplicantIncome = excluded.CoapplicantIncome,
            LoanAmount = excluded.LoanAmount,
            Loan_Status = excluded.Loan_Status,
            TotalIncome = excluded.TotalIncome,
            run_id = excluded.run_id,
            source_file = excluded.source_file,
            loaded_at = excluded.loaded_at
        """)
except sqlite3.DatabaseError as error:
    raise PipelineError("Transactional load failed") from error

print("Environment:", config.environment)
print("Run ID:", run_id)
print("Loaded rows:", connection.execute("SELECT COUNT(*) FROM loan_target").fetchone()[0])

Environment: training
Run ID: 147e4d7d-60a8-4c84-b10d-8080d51ba80a
Loaded rows: 48


### Parameterized SQL

The threshold remains data, not executable SQL. This protects the query and improves plan reuse.

In [5]:
minimum_income = 5000
query = """
SELECT Loan_ID, TotalIncome, Loan_Status
FROM loan_target
WHERE TotalIncome >= ?
ORDER BY TotalIncome DESC
LIMIT ?
"""
high_income = pd.read_sql_query(query, connection, params=(minimum_income, 5))
print(high_income.to_string(index=False))

 Loan_ID  TotalIncome Loan_Status
LP001798      10819.0           Y
LP001813      10383.0           N
LP001814       9703.0           Y
LP001811       7823.0           Y
LP001807       7550.0           Y


### Structured exit codes

A CLI entry point should translate expected failures into stable exit codes while retaining detailed logs for operators.

In [6]:
def pipeline_main(path: Path) -> int:
    try:
        frame = transform(extract(path), str(uuid.uuid4()))
        if frame.empty:
            raise PipelineError("No source rows")
        return 0
    except (ContractError, PipelineError, FileNotFoundError, ValueError):
        return 1

exit_code = pipeline_main(DATA_FILE)
assert exit_code == 0
print("Simulated CLI exit code:", exit_code)

Simulated CLI exit code: 0


## Enterprise Control

Never embed passwords in source code or logs; use environment-specific secret management.

Recommended production controls:

- Retrieve secrets from a managed secret store.
- Rotate credentials and use short-lived identity where available.
- Redact connection strings and parameters from logs.
- Grant the pipeline role only the required schemas and operations.
- Never print environment variables containing credentials.

In [7]:
audit = {
    "run_id": run_id,
    "started_at": transformed["loaded_at"].iloc[0],
    "source_file": DATA_FILE.name,
    "extracted_rows": len(raw),
    "loaded_rows": connection.execute("SELECT COUNT(*) FROM loan_target").fetchone()[0],
    "status": "SUCCESS",
}

assert audit["extracted_rows"] == audit["loaded_rows"]
assert "password" not in " ".join(audit).lower()
print("DE-04 audit record:", audit)
connection.close()

DE-04 audit record: {'run_id': '147e4d7d-60a8-4c84-b10d-8080d51ba80a', 'started_at': '2026-08-26T06:48:00.018062+00:00', 'source_file': 'loan_data_04.csv', 'extracted_rows': 48, 'loaded_rows': 48, 'status': 'SUCCESS'}
